# cap10_wf2020_soxlbacktest_target_contract_v1

Purpose: close the cap10 branch's `wf_2020` target-contract gap without leaking 2020 labels. The existing cap10 joint contract starts in 2020, so it cannot train a `wf_2020` candidate. This notebook derives a fold-local `wf_2020_train` target contract from the qlib-aligned SOXLBacktest replay over pre-2020 data, audits scope/lift/readiness, and records whether a later three-fold BC screen is allowed.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`.
- Direct comparison baseline: `cap10_wf2020_coverage_readiness_v1` and `cap10_foldlocal_joint_target_contract_v1`.
- Role: diagnose.
- Changed layer: `model_selection` / `training_objective`.
- This is not actor retention, PPO promotion, or a runtime overlay.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)
from src.soxl_quant.policy.legacy_soxl import replay_legacy_soxl_policy

EXPERIMENT = "cap10_wf2020_soxlbacktest_target_contract_v1"
CARD_ROOT = ROOT / "runtime" / "model_selection" / "card"
OUT_DIR = CARD_ROOT / EXPERIMENT
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
DRIFT_MAX_DD_CEILING_PP = 2.0
DRIFT_RELATIVE_FLOOR_PP = -5.0

WF2020_FOLD = {
    "name": "wf_2020_train",
    "worker_train_start": "2010-03-11",
    "worker_train_end": "2018-12-31",
    "worker_valid_start": "2019-01-01",
    "worker_valid_end": "2019-12-31",
    "worker_test_start": "2020-01-01",
    "worker_test_end": "2020-12-31",
    # Existing cap10 target contracts include the validation year as pre-test
    # fold-local target material. Keep eval_start at test start to avoid 2020 leakage.
    "train_start": "2010-03-11",
    "train_end": "2019-12-31",
    "eval_start": "2020-01-01",
    "eval_end": "2020-12-31",
}
REQUIRED_WF_WINDOWS = ("wf_2020", "wf_2022", "wf_2023_2024")


In [2]:
def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def safe_rel(path: Path) -> str:
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def max_drawdown_pct(values: pd.Series) -> pd.Series:
    peak = values.cummax()
    return (peak - values) / peak.replace(0.0, np.nan)


def build_wf2020_soxlbacktest_contract(daily: pd.DataFrame, *, fold: dict[str, str]) -> pd.DataFrame:
    frame = daily.copy()
    frame["datetime"] = pd.to_datetime(frame["date"]).dt.normalize()
    frame = frame[
        (frame["datetime"] >= pd.Timestamp(fold["train_start"]))
        & (frame["datetime"] <= pd.Timestamp(fold["train_end"]))
    ].copy()
    if frame.empty:
        return pd.DataFrame()

    frame["strategy_drawdown"] = max_drawdown_pct(pd.to_numeric(frame["strategy_nav"], errors="coerce"))
    frame["position"] = pd.to_numeric(frame["position"], errors="coerce").fillna(0.0).clip(0.0, 1.0)
    frame["position_delta"] = frame["position"].diff().fillna(frame["position"])
    frame["teacher_position"] = frame["position"]

    # Use only train/valid-window state from the qlib-aligned SOXLBacktest replay.
    # Downtrend labels cap high teacher exposure; rebound labels preserve buy/sell timing as a participation floor.
    downtrend = (frame["strategy_drawdown"] >= 0.15) | (
        (pd.to_numeric(frame["panic_score"], errors="coerce") <= -50.0) & (frame["position"] > 0.25)
    )
    rebound = frame["action"].astype(str).str.upper().eq("BUY") | (frame["position_delta"] > 0.04)

    frame["is_downtrend_state"] = downtrend.fillna(False).astype(bool)
    frame["drawdown_control_state"] = frame["is_downtrend_state"]
    frame["is_rebound_cluster"] = rebound.fillna(False).astype(bool)
    frame["release_gate_state"] = frame["is_rebound_cluster"]

    frame["risk_target_position_original"] = np.where(
        frame["is_downtrend_state"],
        np.minimum(frame["teacher_position"], 0.25),
        frame["teacher_position"],
    )
    frame["rebound_target_position"] = np.where(
        frame["is_rebound_cluster"],
        np.maximum(frame["teacher_position"], np.minimum(0.65, frame["teacher_position"] + 0.10)),
        frame["teacher_position"],
    )
    frame["risk_target_position"] = np.where(
        frame["release_gate_state"],
        np.maximum(frame["risk_target_position_original"], frame["rebound_target_position"]),
        frame["risk_target_position_original"],
    )
    frame["objective_weight"] = (
        1.0
        + frame["is_downtrend_state"].astype(float) * 10.0
        + frame["is_rebound_cluster"].astype(float) * 5.0
        + np.clip((frame["strategy_drawdown"] - 0.15) * 40.0, 0.0, 20.0)
    )

    frame["candidate"] = "soxlbacktest_de_risk_teacher"
    frame["window"] = str(fold["name"])
    frame["fold"] = str(fold["name"])
    frame["target_scope"] = "fold_local"
    frame["train_start"] = str(fold["train_start"])
    frame["train_end"] = str(fold["train_end"])
    frame["eval_start"] = str(fold["eval_start"])
    frame["eval_end"] = str(fold["eval_end"])
    frame["source_components"] = "legacy_soxl_backtest,de_risk_overlay"
    frame["source_component_count"] = 2
    frame["source_row_count"] = 1

    cols = [
        "datetime",
        "candidate",
        "window",
        "fold",
        "target_scope",
        "train_start",
        "train_end",
        "eval_start",
        "eval_end",
        "teacher_position",
        "rebound_target_position",
        "risk_target_position_original",
        "risk_target_position",
        "objective_weight",
        "is_rebound_cluster",
        "is_downtrend_state",
        "release_gate_state",
        "drawdown_control_state",
        "source_components",
        "source_component_count",
        "source_row_count",
        "strategy_drawdown",
        "panic_score",
        "action",
        "price",
        "strategy_nav",
        "buyhold_nav",
    ]
    return frame[cols].sort_values("datetime").reset_index(drop=True)


def target_action_distribution(targets: pd.DataFrame) -> pd.DataFrame:
    if targets.empty:
        return pd.DataFrame()
    frame = targets.copy()
    for col in ["teacher_position", "rebound_target_position", "risk_target_position"]:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").fillna(0.0).clip(0.0, 1.0)
    for col in ["is_rebound_cluster", "is_downtrend_state"]:
        frame[col] = frame[col].fillna(False).astype(bool)
    target_position = np.where(
        frame["is_downtrend_state"],
        frame["risk_target_position"],
        np.where(frame["is_rebound_cluster"], frame["rebound_target_position"], frame["teacher_position"]),
    )
    frame["target_action"] = np.round(np.clip(target_position, 0.0, 1.0) / 0.05).astype(int)
    rows = []
    for profile, group in frame.groupby("fold", sort=True):
        counts = group["target_action"].value_counts(normalize=True)
        rows.append(
            {
                "profile": f"{profile}_soxlbacktest_contract",
                "fold": str(profile),
                "samples": int(len(group)),
                "unique_actions": int(group["target_action"].nunique()),
                "dominant_action": int(counts.index[0]),
                "dominant_action_share": float(counts.iloc[0]),
                "action_entropy": float(-(counts * np.log(counts + 1e-12)).sum()),
                "rebound_rows": int(group["is_rebound_cluster"].sum()),
                "downtrend_rows": int(group["is_downtrend_state"].sum()),
            }
        )
    return pd.DataFrame(rows)


In [3]:
source_readiness = load_json(CARD_ROOT / "cap10_wf2020_coverage_readiness_v1" / "summary.json")
source_joint_summary = load_json(CARD_ROOT / "cap10_foldlocal_joint_target_contract_v1" / "summary.json")
source_retraining_handoff = load_json(CARD_ROOT / "retraining_gate_gap_handoff_v1" / "training_objective_handoff.json")

legacy_train = replay_legacy_soxl_policy(
    start=WF2020_FOLD["train_start"],
    end=WF2020_FOLD["train_end"],
)
legacy_test_context = replay_legacy_soxl_policy(
    start=WF2020_FOLD["worker_test_start"],
    end=WF2020_FOLD["worker_test_end"],
)

targets = build_wf2020_soxlbacktest_contract(legacy_train.daily, fold=WF2020_FOLD)
folds = [
    {
        "name": WF2020_FOLD["name"],
        "train_start": WF2020_FOLD["train_start"],
        "train_end": WF2020_FOLD["train_end"],
        "eval_start": WF2020_FOLD["eval_start"],
        "eval_end": WF2020_FOLD["eval_end"],
    }
]

scope_audit = audit_state_objective_target_scope(targets, folds=folds)
scope_summary = summarize_state_objective_target_scope_audit(scope_audit)
lift_audit = audit_state_objective_target_lift(
    targets,
    group_cols=("fold",),
    baseline_risk_position_col="risk_target_position_original",
)
lift_summary = summarize_state_objective_target_lift_audit(lift_audit)
action_distribution = target_action_distribution(targets)
bc_readiness_scores = score_state_objective_bc_readiness(
    action_distribution.drop(columns=["fold"]),
    target_scope="fold_local",
)
bc_readiness_summary = summarize_state_objective_bc_readiness(bc_readiness_scores)

target_summary = targets.groupby("fold", sort=True).agg(
    rows=("datetime", "size"),
    start=("datetime", "min"),
    end=("datetime", "max"),
    rebound_rows=("is_rebound_cluster", "sum"),
    downtrend_rows=("is_downtrend_state", "sum"),
    release_gate_rows=("release_gate_state", "sum"),
    drawdown_control_rows=("drawdown_control_state", "sum"),
    max_objective_weight=("objective_weight", "max"),
).reset_index()

display(target_summary)
display(scope_audit)
display(lift_audit)
display(bc_readiness_scores)


,fold,rows,start,end,rebound_rows,downtrend_rows,release_gate_rows,drawdown_control_rows,max_objective_weight
0,wf_2020_train,2470,2010-03-11,2019-12-31,42,781,42,781,28.328937


,window,target_scope,target_fold,fold,rows,train_overlap_rows,eval_overlap_rows,outside_train_eval_rows,leakage_gate,scope_issue,train_start,train_end,eval_start,eval_end
0,wf_2020_train,fold_local,wf_2020_train,wf_2020_train,2470,2470,0,0,True,,2010-03-11,2019-12-31,2020-01-01,2020-12-31


,fold,rows,rebound_rows,downtrend_rows,rebound_downtrend_rows,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows,mean_current_effective_lift,mean_candidate_release_lift,max_current_effective_lift,max_candidate_release_lift,lift_semantics_gate,release_candidate_gate,target_position_priority,min_lift
0,wf_2020_train,2470,42,781,30,42,42,0.307243,0.307243,0.6498,0.6498,True,True,risk,1.000000e-09


,profile,samples,unique_actions,dominant_action,dominant_action_share,action_entropy,rebound_rows,downtrend_rows,sample_gate,label_distribution_gate,fold_local_scope_gate,bc_readiness_gate,bc_readiness_loss,target_scope
0,wf_2020_train_soxlbacktest_contract,2470,18,5,0.369231,2.031113,42,781,True,True,True,True,0.0,fold_local


In [4]:
source_joint_rows = int((source_joint_summary.get("source_target_rows") or {}).get("joint_contract", 0))
source_folds = [str(value) for value in source_joint_summary.get("folds", [])]
wf2020_target_ready = bool(
    not targets.empty
    and scope_summary.get("training_allowed", False)
    and scope_summary.get("eval_overlap_rows", 1) == 0
    and scope_summary.get("outside_train_eval_rows", 1) == 0
    and lift_summary.get("training_allowed", False)
    and bc_readiness_summary.get("selected_bc_readiness_gate", False)
)

combined_coverage_rows = [
    {
        "fold": "wf_2020",
        "target_fold": WF2020_FOLD["name"],
        "source": EXPERIMENT,
        "target_contract_rows": int(len(targets)),
        "target_contract_ready": bool(wf2020_target_ready),
        "bc_screen_present": False,
        "checkpoint_selection_present": False,
        "ready_for_actor_retention": False,
        "blocking_reason": "missing_wf2020_bc_screen_and_checkpoint_selection" if wf2020_target_ready else "wf2020_target_contract_not_ready",
    },
    {
        "fold": "wf_2022",
        "target_fold": "wf_2022_train",
        "source": "cap10_foldlocal_joint_target_contract_v1",
        "target_contract_rows": source_joint_rows,
        "target_contract_ready": "wf_2022_train" in source_folds,
        "bc_screen_present": True,
        "checkpoint_selection_present": True,
        "ready_for_actor_retention": False,
        "blocking_reason": "checkpoint_target_gate_failed_wf2022",
    },
    {
        "fold": "wf_2023_2024",
        "target_fold": "wf_2023_2024_train",
        "source": "cap10_foldlocal_joint_target_contract_v1",
        "target_contract_rows": source_joint_rows,
        "target_contract_ready": "wf_2023_2024_train" in source_folds,
        "bc_screen_present": True,
        "checkpoint_selection_present": True,
        "ready_for_actor_retention": False,
        "blocking_reason": "wf2020_bc_checkpoint_missing_for_three_fold_retention",
    },
]
combined_coverage = pd.DataFrame(combined_coverage_rows)

missing_target_contract = sorted(
    row["fold"] for row in combined_coverage.to_dict("records") if not bool(row["target_contract_ready"])
)
missing_bc_or_checkpoint = sorted(
    row["fold"]
    for row in combined_coverage.to_dict("records")
    if not bool(row["bc_screen_present"]) or not bool(row["checkpoint_selection_present"])
)

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "reference_baseline": "soxl_backtest_policy_reimplementation_v1",
    "direct_comparison_baseline": [
        "cap10_wf2020_coverage_readiness_v1",
        "cap10_foldlocal_joint_target_contract_v1",
    ],
    "role": "diagnose",
    "changed_layer": ["model_selection", "training_objective"],
    "decision": "wf2020_soxlbacktest_target_contract_ready_for_bc_screen_no_promotion"
    if wf2020_target_ready
    else "wf2020_soxlbacktest_target_contract_blocked_no_training",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_from_this_card": False,
    "bc_screen_allowed_next": bool(wf2020_target_ready),
    "ppo_training_allowed_next": False,
    "target_contract_gate": bool(wf2020_target_ready),
    "target_scope_gate": bool(scope_summary.get("training_allowed", False)),
    "target_lift_gate": bool(lift_summary.get("training_allowed", False)),
    "bc_readiness_gate": bool(bc_readiness_summary.get("selected_bc_readiness_gate", False)),
    "eval_overlap_rows": int(scope_summary.get("eval_overlap_rows", -1)),
    "outside_train_eval_rows": int(scope_summary.get("outside_train_eval_rows", -1)),
    "target_rows": int(len(targets)),
    "rebound_rows": int(targets["is_rebound_cluster"].sum()) if not targets.empty else 0,
    "downtrend_rows": int(targets["is_downtrend_state"].sum()) if not targets.empty else 0,
    "current_positive_rebound_lift_rows": int(lift_summary.get("current_positive_rebound_lift_rows", 0)),
    "selected_samples": int(bc_readiness_summary.get("selected_samples", 0)),
    "selected_unique_actions": int(bc_readiness_summary.get("selected_unique_actions", 0)),
    "selected_dominant_action_share": float(bc_readiness_summary.get("selected_dominant_action_share", 1.0)),
    "worker_window": {
        "fold": "wf_2020",
        "train": [WF2020_FOLD["worker_train_start"], WF2020_FOLD["worker_train_end"]],
        "valid": [WF2020_FOLD["worker_valid_start"], WF2020_FOLD["worker_valid_end"]],
        "test": [WF2020_FOLD["worker_test_start"], WF2020_FOLD["worker_test_end"]],
    },
    "target_window": {
        "target_fold": WF2020_FOLD["name"],
        "train_start": WF2020_FOLD["train_start"],
        "train_end": WF2020_FOLD["train_end"],
        "eval_start": WF2020_FOLD["eval_start"],
        "eval_end": WF2020_FOLD["eval_end"],
        "note": "Pre-2020 train plus validation-year target material only; 2020 is eval/test and has zero target rows.",
    },
    "missing_target_contract_after_this_card": missing_target_contract,
    "missing_bc_or_checkpoint_after_this_card": missing_bc_or_checkpoint,
    "next_mechanism": "run_three_fold_bc_slice_screen_before_any_ppo" if wf2020_target_ready else "revise_wf2020_target_contract",
    "required_next_run_controls": [
        "merge_wf2020_soxlbacktest_contract_with_existing_cap10_joint_contract_for_bc_screen",
        "run_bc_slice_screen_for_wf_2020_wf_2022_wf_2023_2024",
        "keep_bc_initialization_or_iter0_bc_equivalent_as_checkpoint_reference",
        f"reject_ppo_if_ppo_vs_bc_max_dd_delta_pp_gt_{DRIFT_MAX_DD_CEILING_PP:g}",
        f"reject_ppo_if_ppo_vs_bc_relative_delta_pp_lt_{DRIFT_RELATIVE_FLOOR_PP:g}",
        "require_full_fixed_split_and_walkforward_replay_before_actor_retention",
    ],
    "acceptance_gate_for_retention": {
        "required_wf_windows": list(REQUIRED_WF_WINDOWS),
        "max_walkforward_drawdown_pct": 20.0,
        "ytd_return_must_exceed_soxlbacktest_reference": True,
        "ytd_return_must_align_current_best": True,
    },
    "source_context": {
        "prior_decision": source_readiness.get("decision", ""),
        "prior_missing_required_ready_folds": source_readiness.get("missing_required_ready_folds", []),
        "existing_joint_contract_folds": source_folds,
        "retraining_handoff_decision": source_retraining_handoff.get("decision", ""),
    },
    "teacher_replay_metrics": {
        "train_valid_2010_2019": legacy_train.metrics,
        "test_context_2020_not_used_as_label": legacy_test_context.metrics,
    },
    "promotion_blocker": "This card only creates a non-leaking wf_2020 target contract. Actor retention still requires three-fold BC metrics, checkpoint drift selection, fixed-split replay, and YTD return gates.",
    "sidecars": [
        "soxlbacktest_wf2020_target_contract.csv",
        "target_summary.csv",
        "target_scope_audit.csv",
        "target_lift_audit.csv",
        "action_distribution.csv",
        "bc_readiness_scores.csv",
        "combined_coverage_audit.csv",
        "teacher_replay_metrics.json",
        "selection.json",
        "summary.json",
        "conclusion.md",
    ],
}

summary = {
    **selection,
    "conclusion": [
        f"Generated {len(targets)} non-leaking wf_2020 target rows from qlib-aligned SOXLBacktest replay ending {WF2020_FOLD['train_end']}.",
        f"Scope audit has eval_overlap_rows={selection['eval_overlap_rows']} and outside_train_eval_rows={selection['outside_train_eval_rows']}.",
        f"Lift/readiness gates are target_lift_gate={selection['target_lift_gate']} and bc_readiness_gate={selection['bc_readiness_gate']}.",
        "This removes the target-contract gap only; wf_2020 BC and checkpoint selection are still missing, so actor retention remains blocked.",
        "The SOXLBacktest 2020 replay is recorded only as context and was not used for target labels.",
    ],
}

teacher_replay_metrics = {
    "train_valid_2010_2019": legacy_train.metrics,
    "test_context_2020_not_used_as_label": legacy_test_context.metrics,
}

targets.to_csv(OUT_DIR / "soxlbacktest_wf2020_target_contract.csv", index=False)
target_summary.to_csv(OUT_DIR / "target_summary.csv", index=False)
scope_audit.to_csv(OUT_DIR / "target_scope_audit.csv", index=False)
lift_audit.to_csv(OUT_DIR / "target_lift_audit.csv", index=False)
action_distribution.to_csv(OUT_DIR / "action_distribution.csv", index=False)
bc_readiness_scores.to_csv(OUT_DIR / "bc_readiness_scores.csv", index=False)
combined_coverage.to_csv(OUT_DIR / "combined_coverage_audit.csv", index=False)
(OUT_DIR / "teacher_replay_metrics.json").write_text(json.dumps(teacher_replay_metrics, indent=2, sort_keys=True), encoding="utf-8")
(OUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True), encoding="utf-8")
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")

assert selection["eval_overlap_rows"] == 0
assert selection["outside_train_eval_rows"] == 0
assert targets["datetime"].max() < pd.Timestamp("2020-01-01")
assert set(missing_target_contract) == set()

selection


{'experiment': 'cap10_wf2020_soxlbacktest_target_contract_v1',
 'created_at': '2026-06-13T11:35:17.991558+00:00',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
 'direct_comparison_baseline': ['cap10_wf2020_coverage_readiness_v1',
  'cap10_foldlocal_joint_target_contract_v1'],
 'role': 'diagnose',
 'changed_layer': ['model_selection', 'training_objective'],
 'decision': 'wf2020_soxlbacktest_target_contract_ready_for_bc_screen_no_promotion',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'bc_screen_allowed_next': True,
 'ppo_training_allowed_next': False,
 'target_contract_gate': True,
 'target_scope_gate': True,
 'target_lift_gate': True,
 'bc_readiness_gate': True,
 'eval_overlap_rows': 0,
 'outside_train_eval_rows': 0,
 'target_rows': 2470,
 'rebound_rows': 42,
 'downtrend_rows': 781,
 'current_positive_rebound_lift_rows': 42,
 'selected_sam

In [5]:
conclusion = f"""# {EXPERIMENT}

- Decision: `{selection['decision']}`.
- `wf_2020` target rows: `{selection['target_rows']}` from qlib-aligned SOXLBacktest replay, with target dates `{targets['datetime'].min().date()}` to `{targets['datetime'].max().date()}`.
- Leakage audit: `eval_overlap_rows={selection['eval_overlap_rows']}`, `outside_train_eval_rows={selection['outside_train_eval_rows']}`; no 2020 rows enter the target contract.
- Label health: `rebound_rows={selection['rebound_rows']}`, `downtrend_rows={selection['downtrend_rows']}`, `unique_actions={selection['selected_unique_actions']}`, dominant action share `{selection['selected_dominant_action_share']:.3f}`.
- Coverage effect: target-contract coverage now exists for `wf_2020`, `wf_2022`, and `wf_2023_2024`, but `wf_2020` still lacks BC slice metrics and checkpoint selection.
- Retention remains blocked: no actor, PPO checkpoint, runtime overlay, or production model is promoted from this notebook.
- Next action: `{selection['next_mechanism']}` with drift stop `PPO-vs-BC MaxDD delta <= {DRIFT_MAX_DD_CEILING_PP:.1f}pp` and relative delta `>= {DRIFT_RELATIVE_FLOOR_PP:.1f}pp` before any full retraining continuation.
"""
(OUT_DIR / "conclusion.md").write_text(conclusion, encoding="utf-8")
display(Markdown(conclusion))


# cap10_wf2020_soxlbacktest_target_contract_v1

- Decision: `wf2020_soxlbacktest_target_contract_ready_for_bc_screen_no_promotion`.
- `wf_2020` target rows: `2470` from qlib-aligned SOXLBacktest replay, with target dates `2010-03-11` to `2019-12-31`.
- Leakage audit: `eval_overlap_rows=0`, `outside_train_eval_rows=0`; no 2020 rows enter the target contract.
- Label health: `rebound_rows=42`, `downtrend_rows=781`, `unique_actions=18`, dominant action share `0.369`.
- Coverage effect: target-contract coverage now exists for `wf_2020`, `wf_2022`, and `wf_2023_2024`, but `wf_2020` still lacks BC slice metrics and checkpoint selection.
- Retention remains blocked: no actor, PPO checkpoint, runtime overlay, or production model is promoted from this notebook.
- Next action: `run_three_fold_bc_slice_screen_before_any_ppo` with drift stop `PPO-vs-BC MaxDD delta <= 2.0pp` and relative delta `>= -5.0pp` before any full retraining continuation.
